# 그래프 직접 생성

- 이 실습에서는 LLM을 사용하지 않습니다. 사람이 작성한 Cypher를 `Neo4jGraph.query()`로 실행합니다.

## 1. Neo4jGraph 연결 정보

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print("URI:", URI or "(미설정)")
print("DATABASE:", DATABASE)

URI: bolt://localhost:7687
DATABASE: neo4j


## 2. 온라인 쇼핑몰 노드

- 직접 작성한 Cypher와 다음 노트북의 LLM 적재가 같은 `타입별 라벨 + id` 구조를 사용합니다.
- 공통 `Entity` 라벨을 사용하지 않으므로 Neo4j Browser에서 타입마다 다른 색상으로 표시됩니다.

In [2]:
# 노드 생성 쿼리. MERGE: 같은 라벨과 id 이미 있으면 그 노드를 재사용해 셀을 여러번 실행해도 중복 노드가 늘어나지 않게 함.
# MERGE: 같은 라벨과 id 이미 있으면 그 노드를 재사용해 셀을 여러번 실행해도 중복 노드가 늘어나지 않게 함.
# SET: 신규/기존 노드 모두에 표시명과 도메인 타입 속성을 맞춤

NODES_CYPHER = """
MERGE (customer_a:Customer {id: "고객 A"})
SET customer_a.name = "고객 A", customer_a.type = "Customer"

MERGE (customer_b:Customer {id: "고객 B"})
SET customer_b.name = "고객 B", customer_b.type = "Customer"

MERGE (order_1001:Order {id: "주문 O-1001"})
SET order_1001.name = "주문 O-1001", order_1001.type = "Order"

MERGE (order_1002:Order {id: "주문 O-1002"})
SET order_1002.name = "주문 O-1002", order_1002.type = "Order"

MERGE (earphones:Product {id: "무선 이어폰"})
SET earphones.name = "무선 이어폰", earphones.type = "Product"

MERGE (smart_watch:Product {id: "스마트 워치"})
SET smart_watch.name = "스마트 워치", smart_watch.type = "Product"

MERGE (laptop_pouch:Product {id: "노트북 파우치"})
SET laptop_pouch.name = "노트북 파우치", laptop_pouch.type = "Product"

MERGE (icheon:Warehouse {id: "이천 물류센터"})
SET icheon.name = "이천 물류센터", icheon.type = "Warehouse"

MERGE (busan:Warehouse {id: "부산 물류센터"})
SET busan.name = "부산 물류센터", busan.type = "Warehouse"

MERGE (fast_delivery:Courier {id: "빠른택배"})
SET fast_delivery.name = "빠른택배", fast_delivery.type = "Courier"

MERGE (safe_delivery:Courier {id: "안심택배"})
SET safe_delivery.name = "안심택배", safe_delivery.type = "Courier"

MERGE (summer_coupon:Coupon {id: "여름할인 쿠폰"})
SET summer_coupon.name = "여름할인 쿠폰", summer_coupon.type = "Coupon"

MERGE (welcome_coupon:Coupon {id: "신규가입 쿠폰"})
SET welcome_coupon.name = "신규가입 쿠폰", welcome_coupon.type = "Coupon"

RETURN
    customer_a, customer_b,
    order_1001, order_1002,
    earphones, smart_watch, laptop_pouch,
    icheon, busan,
    fast_delivery, safe_delivery,
    summer_coupon, welcome_coupon
""".strip()

print(NODES_CYPHER)

MERGE (customer_a:Customer {id: "고객 A"})
SET customer_a.name = "고객 A", customer_a.type = "Customer"

MERGE (customer_b:Customer {id: "고객 B"})
SET customer_b.name = "고객 B", customer_b.type = "Customer"

MERGE (order_1001:Order {id: "주문 O-1001"})
SET order_1001.name = "주문 O-1001", order_1001.type = "Order"

MERGE (order_1002:Order {id: "주문 O-1002"})
SET order_1002.name = "주문 O-1002", order_1002.type = "Order"

MERGE (earphones:Product {id: "무선 이어폰"})
SET earphones.name = "무선 이어폰", earphones.type = "Product"

MERGE (smart_watch:Product {id: "스마트 워치"})
SET smart_watch.name = "스마트 워치", smart_watch.type = "Product"

MERGE (laptop_pouch:Product {id: "노트북 파우치"})
SET laptop_pouch.name = "노트북 파우치", laptop_pouch.type = "Product"

MERGE (icheon:Warehouse {id: "이천 물류센터"})
SET icheon.name = "이천 물류센터", icheon.type = "Warehouse"

MERGE (busan:Warehouse {id: "부산 물류센터"})
SET busan.name = "부산 물류센터", busan.type = "Warehouse"

MERGE (fast_delivery:Courier {id: "빠른택배"})
SET fast_delivery.name = "빠른택배", fast

## 3. 온라인 쇼핑몰 관계


```text
고객 A → 주문 O-1001 → 무선 이어폰, 노트북 파우치
                     ├→ 이천 물류센터
                     ├→ 빠른택배
                     └→ 여름할인 쿠폰

고객 B → 주문 O-1002 → 스마트 워치
                     ├→ 부산 물류센터
                     ├→ 안심택배
                     └→ 신규가입 쿠폰
```

In [3]:
RELATIONSHIPS_CYPHER = """
MATCH
    (customer_a:Customer {id: "고객 A"}),
    (customer_b:Customer {id: "고객 B"}),
    (order_1001:Order {id: "주문 O-1001"}),
    (order_1002:Order {id: "주문 O-1002"}),
    (earphones:Product {id: "무선 이어폰"}),
    (smart_watch:Product {id: "스마트 워치"}),
    (laptop_pouch:Product {id: "노트북 파우치"}),
    (icheon:Warehouse {id: "이천 물류센터"}),
    (busan:Warehouse {id: "부산 물류센터"}),
    (fast_delivery:Courier {id: "빠른택배"}),
    (safe_delivery:Courier {id: "안심택배"}),
    (summer_coupon:Coupon {id: "여름할인 쿠폰"}),
    (welcome_coupon:Coupon {id: "신규가입 쿠폰"})

MERGE (customer_a)-[:PLACED]->(order_1001)
MERGE (customer_b)-[:PLACED]->(order_1002)
MERGE (order_1001)-[:CONTAINS]->(earphones)
MERGE (order_1001)-[:CONTAINS]->(laptop_pouch)
MERGE (order_1002)-[:CONTAINS]->(smart_watch)
MERGE (order_1001)-[:FULFILLED_BY]->(icheon)
MERGE (order_1002)-[:FULFILLED_BY]->(busan)
MERGE (order_1001)-[:SHIPPED_BY]->(fast_delivery)
MERGE (order_1002)-[:SHIPPED_BY]->(safe_delivery)
MERGE (order_1001)-[:USES_COUPON]->(summer_coupon)
MERGE (order_1002)-[:USES_COUPON]->(welcome_coupon)

RETURN
    customer_a, customer_b,
    order_1001, order_1002,
    earphones, smart_watch, laptop_pouch,
    icheon, busan,
    fast_delivery, safe_delivery,
    summer_coupon, welcome_coupon
""".strip()

print(RELATIONSHIPS_CYPHER)

MATCH
    (customer_a:Customer {id: "고객 A"}),
    (customer_b:Customer {id: "고객 B"}),
    (order_1001:Order {id: "주문 O-1001"}),
    (order_1002:Order {id: "주문 O-1002"}),
    (earphones:Product {id: "무선 이어폰"}),
    (smart_watch:Product {id: "스마트 워치"}),
    (laptop_pouch:Product {id: "노트북 파우치"}),
    (icheon:Warehouse {id: "이천 물류센터"}),
    (busan:Warehouse {id: "부산 물류센터"}),
    (fast_delivery:Courier {id: "빠른택배"}),
    (safe_delivery:Courier {id: "안심택배"}),
    (summer_coupon:Coupon {id: "여름할인 쿠폰"}),
    (welcome_coupon:Coupon {id: "신규가입 쿠폰"})

MERGE (customer_a)-[:PLACED]->(order_1001)
MERGE (customer_b)-[:PLACED]->(order_1002)
MERGE (order_1001)-[:CONTAINS]->(earphones)
MERGE (order_1001)-[:CONTAINS]->(laptop_pouch)
MERGE (order_1002)-[:CONTAINS]->(smart_watch)
MERGE (order_1001)-[:FULFILLED_BY]->(icheon)
MERGE (order_1002)-[:FULFILLED_BY]->(busan)
MERGE (order_1001)-[:SHIPPED_BY]->(fast_delivery)
MERGE (order_1002)-[:SHIPPED_BY]->(safe_delivery)
MERGE (order_1001)-[:USES_COUPON]->(summ

## 4. 기존 `Entity` 라벨 마이그레이션

- 이전 실습에서 만든 노드와 관계는 그대로 유지하면서 공통 `Entity` 라벨만 제거합니다.
- 각 노드의 `type` 속성에 맞는 타입별 라벨을 먼저 보장하므로 기존 데이터에도 안전하게 적용할 수 있습니다.

In [4]:
LABEL_MIGRATION_CYPHER = """
MATCH (entity:Entity)
WHERE entity.type IN [
    "Customer", "Order", "Product",
    "Warehouse", "Courier", "Coupon"
]
FOREACH (_ IN CASE WHEN entity.type = "Customer" THEN [1] ELSE [] END |
    SET entity:Customer)
FOREACH (_ IN CASE WHEN entity.type = "Order" THEN [1] ELSE [] END |
    SET entity:Order)
FOREACH (_ IN CASE WHEN entity.type = "Product" THEN [1] ELSE [] END |
    SET entity:Product)
FOREACH (_ IN CASE WHEN entity.type = "Warehouse" THEN [1] ELSE [] END |
    SET entity:Warehouse)
FOREACH (_ IN CASE WHEN entity.type = "Courier" THEN [1] ELSE [] END |
    SET entity:Courier)
FOREACH (_ IN CASE WHEN entity.type = "Coupon" THEN [1] ELSE [] END |
    SET entity:Coupon)
REMOVE entity:Entity
""".strip()

print(LABEL_MIGRATION_CYPHER)

MATCH (entity:Entity)
WHERE entity.type IN [
    "Customer", "Order", "Product",
    "Warehouse", "Courier", "Coupon"
]
FOREACH (_ IN CASE WHEN entity.type = "Customer" THEN [1] ELSE [] END |
    SET entity:Customer)
FOREACH (_ IN CASE WHEN entity.type = "Order" THEN [1] ELSE [] END |
    SET entity:Order)
FOREACH (_ IN CASE WHEN entity.type = "Product" THEN [1] ELSE [] END |
    SET entity:Product)
FOREACH (_ IN CASE WHEN entity.type = "Warehouse" THEN [1] ELSE [] END |
    SET entity:Warehouse)
FOREACH (_ IN CASE WHEN entity.type = "Courier" THEN [1] ELSE [] END |
    SET entity:Courier)
FOREACH (_ IN CASE WHEN entity.type = "Coupon" THEN [1] ELSE [] END |
    SET entity:Coupon)
REMOVE entity:Entity


## 5. `Neo4jGraph.query()` 실행

In [10]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=URI,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    refresh_schema=False  # 👈 APOC 메타데이터 자동 조회를 비활성화합니다.
)

graph.query(LABEL_MIGRATION_CYPHER)
graph.query(NODES_CYPHER)
graph.query(RELATIONSHIPS_CYPHER)

[{'customer_a': {'name': '고객 A', 'id': '고객 A', 'type': 'Customer'},
  'customer_b': {'name': '고객 B', 'id': '고객 B', 'type': 'Customer'},
  'order_1001': {'name': '주문 O-1001', 'id': '주문 O-1001', 'type': 'Order'},
  'order_1002': {'name': '주문 O-1002', 'id': '주문 O-1002', 'type': 'Order'},
  'earphones': {'name': '무선 이어폰', 'id': '무선 이어폰', 'type': 'Product'},
  'smart_watch': {'name': '스마트 워치', 'id': '스마트 워치', 'type': 'Product'},
  'laptop_pouch': {'name': '노트북 파우치', 'id': '노트북 파우치', 'type': 'Product'},
  'icheon': {'name': '이천 물류센터', 'id': '이천 물류센터', 'type': 'Warehouse'},
  'busan': {'name': '부산 물류센터', 'id': '부산 물류센터', 'type': 'Warehouse'},
  'fast_delivery': {'name': '빠른택배', 'id': '빠른택배', 'type': 'Courier'},
  'safe_delivery': {'name': '안심택배', 'id': '안심택배', 'type': 'Courier'},
  'summer_coupon': {'name': '여름할인 쿠폰', 'id': '여름할인 쿠폰', 'type': 'Coupon'},
  'welcome_coupon': {'name': '신규가입 쿠폰', 'id': '신규가입 쿠폰', 'type': 'Coupon'}}]

## 6. 생성 결과 조회

In [7]:
READ_GRAPH_CYPHER = """
MATCH (source)-[relation]->(target)
RETURN source.id AS source,
       labels(source) AS source_labels,
       type(relation) AS relation,
       target.id AS target,
       labels(target) AS target_labels
ORDER BY source, relation, target
""".strip()

In [11]:
if graph is not None:
    rows = graph.query(READ_GRAPH_CYPHER)
    for row in rows:
        print(row)

    else:
        print(READ_GRAPH_CYPHER)

{'source': '고객 A', 'source_labels': ['Customer'], 'relation': 'PLACED', 'target': '주문 O-1001', 'target_labels': ['Order']}
{'source': '고객 B', 'source_labels': ['Customer'], 'relation': 'PLACED', 'target': '주문 O-1002', 'target_labels': ['Order']}
{'source': '주문 O-1001', 'source_labels': ['Order'], 'relation': 'CONTAINS', 'target': '노트북 파우치', 'target_labels': ['Product']}
{'source': '주문 O-1001', 'source_labels': ['Order'], 'relation': 'CONTAINS', 'target': '무선 이어폰', 'target_labels': ['Product']}
{'source': '주문 O-1001', 'source_labels': ['Order'], 'relation': 'FULFILLED_BY', 'target': '이천 물류센터', 'target_labels': ['Warehouse']}
{'source': '주문 O-1001', 'source_labels': ['Order'], 'relation': 'SHIPPED_BY', 'target': '빠른택배', 'target_labels': ['Courier']}
{'source': '주문 O-1001', 'source_labels': ['Order'], 'relation': 'USES_COUPON', 'target': '여름할인 쿠폰', 'target_labels': ['Coupon']}
{'source': '주문 O-1002', 'source_labels': ['Order'], 'relation': 'CONTAINS', 'target': '스마트 워치', 'target_labels': 

## 7. Neo4j Browser에서 보기

```cypher
MATCH path=(source)-[relation]->(target)
RETURN path
```

노드를 드래그하고 타입별 색상과 관계의 화살표 방향을 확인하세요.

## 8. 삭제

In [12]:
graph.query(
    """
    MATCH (n)
    DETACH DELETE n;    
    """
)

[]